# 공용 학습 노트북 (Colab · Kaggle · VS Code 원격)

- **노트북은 이것 하나만 씁니다.** 실험마다 복사하지 않고, 셀 1의 `BRANCH`, `EXP_YAML` 두 줄만 바꾸세요. 바꾼 내용은 커밋하지 마세요.
- **학습은 마지막 셀의 `scripts/train.py`가 합니다.** 그 앞의 셀은 세션마다 반복해야 하는 준비(clone, 설치, 키, 데이터)입니다.
- 새 세션이든 같은 세션이든 **Run All**하면 됩니다. 이미 끝난 준비는 빠르게 지나갑니다.
- 원격세션(Colab, Kaggle VM)에서는 GitHub의 브랜치를 clone해서 실행합니다. **실험 YAML을 먼저 push해놓고 실행하세요.**

| 실행 방식 | 실행 전에 할 일 |
| -- | -- |
| 브라우저 Colab | 런타임 → 런타임 유형 변경 → T4 GPU. 왼쪽 🔑 보안 비밀에 `WANDB_API_KEY` 추가, 노트북 액세스 켜기 |
| VS Code + Colab 확장 | Select Kernel → Colab → New Colab Server(이미 있으면 있는 거 사용) → GPU T4. 키 셀에서 입력창이 뜨면 W&B 키를 붙여 넣기 |
| 브라우저 Kaggle | Settings → Accelerator **GPU T4 x2** (P100 금지), Internet On, Add-ons → Secrets `WANDB_API_KEY`, Add Input → 팀 원본 데이터셋 |
| VS Code + Kaggle Jupyter Server | 위 Kaggle 설정을 웹에서 먼저 한 뒤, Run → Kaggle Jupyter Server의 VS Code Compatible URL을 Select Kernel → Existing Jupyter Server에 붙여 넣기 |

> 키는 셀에 직접 쓰지 않습니다. 레포가 공개되어 있습니다. 이 노트북을 커밋할 때는 **출력을 모두 지운 뒤** 커밋합니다.

In [ ]:
# ── 실험마다 바꾸는 값 ──
BRANCH = "main"                  # 실험 YAML을 push한 브랜치. 예) "exp/yolov8s-mosaic"
EXP_YAML = "configs/smoke.yaml"  # 예) "configs/exp/yolov8s_640.yaml"

# ── 팀 고정값 (정해지면 한 번만 고침) ──
DRIVE_RAW = "/content/drive/MyDrive/pill-data/raw.zip"  # Colab: 각자 Drive에 올린 원본 zip 또는 폴더
KAGGLE_INPUT = "/kaggle/input"                          # Kaggle: 이 아래에서 train_annotations 폴더를 찾음
WANDB_ENTITY = ""                                       # W&B 팀을 쓰기로 하면 팀 이름. 비우면 각자 개인 계정
REPO_URL = "https://github.com/codeit-ai-15th/pill-object-detection.git"

In [ ]:
# 실행 환경 확인
import os
import subprocess
import sys
from pathlib import Path


def sh(cmd):
    """셸 명령을 실행해 출력을 보여 주고, 실패하면 노트북 실행을 멈춘다."""
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"명령이 실패했습니다 (종료 코드 {proc.returncode}): {cmd}")


if "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules:
    ENV, REPO = "colab", Path("/content/pill-object-detection")
elif "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    # /kaggle/working은 버전 저장 때 함께 저장되고 20GB 제한이 있어서, 레포와 .venv는 그 밖에 둔다
    ENV, REPO = "kaggle", Path("/kaggle/tmp/pill-object-detection")
else:
    ENV = "local"
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.environ["UV_FROZEN"] = "1"        # uv.lock에 적힌 버전 그대로 설치
os.environ["UV_LINK_MODE"] = "copy"  # uv 캐시와 .venv가 다른 디스크여도 경고 없이 설치
if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
print("실행 환경:", ENV, "| 레포 위치:", REPO)

In [ ]:
# GPU 확인: T4가 아니면 멈춘다 (로컬은 건너뜀)
if ENV != "local":
    sh("nvidia-smi")  # 오른쪽 위 CUDA Version이 13.0 이상이어야 우리 torch(cu130)가 GPU를 쓸 수 있다
    gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader",
                         shell=True, capture_output=True, text=True).stdout
    if "T4" not in gpu:
        raise RuntimeError("T4 GPU가 아닙니다. Colab: 런타임 → 런타임 유형 변경 → T4 GPU / "
                           "Kaggle: Settings → Accelerator → GPU T4 x2 (P100은 쓸 수 없음)")

In [ ]:
# 레포 받기: 원격에는 로컬 파일이 없으므로 GitHub에서 BRANCH를 받는다
if ENV != "local":
    if not (REPO / ".git").exists():
        REPO.parent.mkdir(parents=True, exist_ok=True)
        sh(f"git clone -b {BRANCH} {REPO_URL} {REPO}")
    else:  # 같은 세션에서 다시 실행: 원격 BRANCH의 최신 상태로 맞춤
        sh(f"git -C {REPO} fetch origin {BRANCH} && git -C {REPO} checkout -f -B {BRANCH} origin/{BRANCH}")
os.chdir(REPO)
sh("git log --oneline -1")

In [ ]:
# 패키지 설치: 팀원 모두 uv.lock에 적힌 같은 버전을 쓴다 (처음 세션은 torch를 받느라 몇 분 걸림)
if ENV != "local":
    sh("pip install -q uv")  # uv 자체를 설치할 때만 pip을 쓴다. 프로젝트 패키지는 모두 uv sync로 설치
sh("uv sync")
if ENV != "local":
    sh("""uv run python - <<'EOF'
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda)
if not torch.cuda.is_available():
    raise SystemExit("GPU를 쓸 수 없습니다. nvidia-smi의 CUDA Version이 13.0보다 낮은지 확인하세요.")
torch.zeros(1).cuda()  # 실제로 GPU에서 계산되는지 확인 (지원하지 않는 GPU면 여기서 실패)
print("GPU", torch.cuda.get_device_name(0), "| 지원 아키텍처", torch.cuda.get_arch_list())
EOF""")

In [ ]:
# W&B 키: 환경마다 있는 곳에서 찾아 환경 변수로 넣는다. 값은 출력하지 않는다.
import getpass


def load_secret(name):
    """키를 환경 변수로 넣고, 어디서 찾았는지 돌려준다."""
    if os.environ.get(name):
        return "이미 설정된 환경 변수"
    try:  # 브라우저 Colab. VS Code Colab 확장에서는 오류가 나서 다음 방법으로 넘어간다
        from google.colab import userdata
        os.environ[name] = userdata.get(name)
        return "Colab 보안 비밀"
    except Exception:
        pass
    try:  # 브라우저 Kaggle, Kaggle Jupyter Server
        from kaggle_secrets import UserSecretsClient
        os.environ[name] = UserSecretsClient().get_secret(name)
        return "Kaggle Secrets"
    except Exception:
        pass
    for env_file in (Path("/content/.env"), REPO / ".env"):  # train.py가 이 .env를 직접 읽는다
        if env_file.exists() and any(line.startswith(f"{name}=") and line.split("=", 1)[1].strip()
                                     for line in env_file.read_text().splitlines()):
            return f"{env_file} (train.py가 읽음)"
    os.environ[name] = getpass.getpass(f"{name}을 붙여 넣으세요 (화면에 보이지 않음): ").strip()
    return "직접 입력"


if os.environ.get("WANDB_MODE") in ("offline", "disabled"):
    print("WANDB_MODE =", os.environ["WANDB_MODE"], "→ 키 없이 진행")
else:
    print("WANDB_API_KEY:", load_secret("WANDB_API_KEY"))

In [ ]:
# 원본 데이터 배치 → YOLO 형식으로 변환
if ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    if DRIVE_RAW.endswith(".zip"):
        if not Path("/content/raw").exists():  # Drive에서 바로 읽으면 느려서 VM 디스크에 푼다
            sh(f'unzip -q "{DRIVE_RAW}" -d /content/raw')
        search_root = Path("/content/raw")
    else:
        search_root = Path(DRIVE_RAW)
elif ENV == "kaggle":
    search_root = Path(KAGGLE_INPUT)
else:
    search_root = REPO / "data" / "raw"

found = sorted(search_root.rglob("train_annotations"))
if not found:
    raise RuntimeError(f"{search_root} 아래에서 train_annotations 폴더를 찾지 못했습니다. 셀 1의 데이터 경로를 확인하세요.")
RAW_DIR = found[0].parent
print("원본 데이터:", RAW_DIR)
sh(f'uv run python scripts/prepare_data.py --raw-dir "{RAW_DIR}"')

In [ ]:
# 학습
!uv run python scripts/train.py {EXP_YAML}

## 결과 확인
- 위 출력 마지막 부분의 W&B 링크에서 epoch별 loss, mAP, **mAP75-95(대회 지표)**를 봅니다.
- 세션이 끝나면 `runs/`는 사라지지만 W&B의 기록과 best.pt(Artifacts 탭)는 남습니다.
- Kaggle에서 몇 시간 걸리는 학습은 **Save Version → Save & Run All**로 돌리면 브라우저를 닫아도 계속됩니다.
- PR 설명에 W&B run 링크와 mAP75-95를 적습니다.